# ***Part 3 — Machine Learning: predict no-shows and choose the model honestly***

In [ ]:
#Task 3.1 — Features, split and baseline
import pandas as pd
from sklearn.metrics import accuracy_score
from sklearn.model_selection import train_test_split

# Load the cleaned dataset without changing row order
df = pd.read_csv("appointments_clean.csv")

# Select features and encode categoricals
#Build X from age, distance_km, fee_inr, lead_days, prior_no_shows, reminder_sent
cols = [
    "age",
    "distance_km",
    "fee_inr",
    "lead_days",
    "prior_no_shows",
    "reminder_sent",
    "booking_channel",
    "visit_type",
]
X = df[cols].copy()
X["reminder_sent"] = X["reminder_sent"].map({"Yes": 1, "No": 0})
#the one-hot encoded booking_channel
#visit_type (pd.get_dummies(X, columns=["booking_channel","visit_type"]))
X = pd.get_dummies(X, columns=["booking_channel", "visit_type"], dtype=int)
#Target y = no_show
y = df["no_show"]

# Split with train_test_split(X, y, test_size=0.2, random_state=42)
X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.2, random_state=42
)

# Baseline: always predict "Attended" (0)
baseline_pred = [0] * len(y_test)

#Report the training and test sizes, the number of no-shows in the test set
#the baseline accuracy of always predicting "Attended"
baseline_acc = accuracy_score(y_test, baseline_pred)
print("number of column:",X.shape[1])
print("Train size:", len(X_train))
print("Test size:", len(X_test))
print("Test no-shows:", int(y_test.sum()))
print("Baseline accuracy:", round(baseline_acc, 4))

number of column: 11
Train size: 792
Test size: 198
Test no-shows: 49
Baseline accuracy: 0.7525


## Task 3.1 — Feature Encoding, Train/Test Split & Majority-Class Baseline
- Load `appointments_clean.csv`, map `reminder_sent` (`Yes` -> `1`, `No` -> `0`), and one-hot encode `booking_channel` and `visit_type` via `pd.get_dummies()`.
- Split `X` and `y` (`no_show`) into an 80/20 train/test split (`random_state=42`) and compute the baseline accuracy of always predicting `"Attended"` (`0`)[span_0](start_span)[span_0](end_span)[span_1](start_span)[span_1](end_span).
-

In [ ]:
#Task 3.2 — Logistic regression
import numpy as np
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import (
    accuracy_score,
    confusion_matrix,
    f1_score,
    precision_score,
    recall_score,
    roc_auc_score,
)
from sklearn.preprocessing import StandardScaler

# Scale features using training data only
scaler = StandardScaler()
X_train_s = scaler.fit_transform(X_train)
X_test_s = scaler.transform(X_test)

# Train Logistic Regression
model = LogisticRegression(max_iter=1000)
model.fit(X_train_s, y_train)

# Predict on test set
y_pred = model.predict(X_test_s)
y_prob = model.predict_proba(X_test_s)[:, 1]

print("Accuracy:", round(accuracy_score(y_test, y_pred), 4))
print("Precision:", round(precision_score(y_test, y_pred), 4))
print("Recall:", round(recall_score(y_test, y_pred), 4))
print("F1:", round(f1_score(y_test, y_pred), 4))
print("ROC-AUC:", round(roc_auc_score(y_test, y_prob), 4))
print("\nConfusion Matrix:\n", confusion_matrix(y_test, y_pred))

# Print coefficients next to feature names
print("\nFeature Coefficients:")
for name, coef in zip(X.columns, model.coef_[0]):
    print(f"{name:25s} {coef:.4f}")

# Odds ratio for reminder_sent (scaled vs actual 0 -> 1 change)
idx = list(X.columns).index("reminder_sent")
coef_rem = model.coef_[0][idx]
std_rem = scaler.scale_[idx]

exp_scaled = np.exp(coef_rem)
exp_unscaled = np.exp(coef_rem / std_rem)

print(f"\nexp(coef) on scaled feature: {exp_scaled:.4f}")
print(f"exp(coef / std) for No -> Yes: {exp_unscaled:.4f}")

Accuracy: 0.7879
Precision: 0.6296
Recall: 0.3469
F1: 0.4474
ROC-AUC: 0.7889

Confusion Matrix:
 [[139  10]
 [ 32  17]]

Feature Coefficients:
age                       -0.2539
distance_km               0.2534
fee_inr                   0.4528
lead_days                 0.6145
prior_no_shows            0.6496
reminder_sent             -0.5807
booking_channel_App       -0.0468
booking_channel_Phone     0.2159
booking_channel_Walk-In   -0.1864
visit_type_Consultation   0.0157
visit_type_Follow-up      -0.0157

exp(coef) on scaled feature: 0.5595
exp(coef / std) for No -> Yes: 0.3094


## Task 3.2 — Logistic Regression & Odds-Ratio Analysis
- Fit `StandardScaler` on `X_train` only, transform both splits, and train `LogisticRegression(max_iter=1000)` to evaluate Accuracy, Precision, Recall, F1, ROC-AUC, and the confusion matrix.
- Inspect feature coefficients and compute the odds ratio (`exp(coef)` and `exp(coef / std)`) for `reminder_sent` to quantify how sending a reminder shifts no-show odds.
-

Takeaway (Task 3.2): The two values differ because exp(coef) = 0.5595 reflects a one-standard-deviation increase in the standardized reminder_sent feature ($\text{SD} \approx 0.49$), whereas exp(coef / std) = 0.3094 converts that effect back to the raw binary change from No (0) to Yes (1). In this fitted model, holding the other features constant, appointments with a reminder sent are associated with roughly 69% lower odds of a no-show—an observational association, not proof of causation.

In [ ]:
#Task 3.3 — Decision tree and overfitting
from sklearn.metrics import accuracy_score
from sklearn.tree import DecisionTreeClassifier

# 1. Full-depth Decision Tree (no depth limit)
dt_full = DecisionTreeClassifier(random_state=42)
dt_full.fit(X_train, y_train)

print("=== Full-Depth Decision Tree ===")
print("Depth:", dt_full.get_depth())
print(
    "Train accuracy:",
    round(accuracy_score(y_train, dt_full.predict(X_train)), 4),
)
print(
    "Test accuracy: ", round(accuracy_score(y_test, dt_full.predict(X_test)), 4)
)

# 2. Pruned Decision Tree (max_depth=3)
dt_3 = DecisionTreeClassifier(max_depth=3, random_state=42)
dt_3.fit(X_train, y_train)

print("\n=== Depth-3 Decision Tree ===")
print("Depth:", dt_3.get_depth())
print(
    "Train accuracy:", round(accuracy_score(y_train, dt_3.predict(X_train)), 4)
)
print("Test accuracy: ", round(accuracy_score(y_test, dt_3.predict(X_test)), 4))

=== Full-Depth Decision Tree ===
Depth: 15
Train accuracy: 1.0
Test accuracy:  0.6717

=== Depth-3 Decision Tree ===
Depth: 3
Train accuracy: 0.822
Test accuracy:  0.7778


## Task 3.3 — Decision Tree & Overfitting Analysis
- Train an unpruned `DecisionTreeClassifier(random_state=42)` (depth 15) alongside a pruned `DecisionTreeClassifier(max_depth=3, random_state=42)` to compare train vs. test accuracy.
- Highlight how the full-depth tree memorizes training data (`1.0000` train vs. `0.6717` test accuracy), whereas pruning to `max_depth=3` curbs overfitting and lifts test accuracy to `0.7778`.
-

Takeaway (Task 3.3): The gap between 1.0000 training accuracy and 0.6717 test accuracy (below the 0.7525 baseline) shows that the unpruned depth-15 decision tree severely overfit and memorized training noise, whereas pruning to max_depth=3 prevented memorization and raised test accuracy to 0.7778.

In [ ]:
#Task 3.4 — Random forest and a comparison table
import pandas as pd
from sklearn.ensemble import RandomForestClassifier
from sklearn.metrics import accuracy_score, recall_score, roc_auc_score

# 1. Fit Random Forest
#Report accuracy, recall and ROC-AUC (use predict_proba(X_test)[:, 1] for AUC)

rf = RandomForestClassifier(n_estimators=100, random_state=42)
rf.fit(X_train, y_train)

rf_pred = rf.predict(X_test)
rf_prob = rf.predict_proba(X_test)[:, 1]

print("=== Random Forest Metrics ===")
print("Train accuracy:", round(accuracy_score(y_train, rf.predict(X_train)), 4))
print("Test accuracy: ", round(accuracy_score(y_test, rf_pred), 4))
print("Recall:        ", round(recall_score(y_test, rf_pred), 4))
print("ROC-AUC:       ", round(roc_auc_score(y_test, rf_prob), 4))

# 2. Top three feature importances
importances = pd.Series(rf.feature_importances_, index=X.columns).sort_values(
    ascending=False
)
print("\n=== Top 3 Feature Importances ===")
print(importances.head(3).round(4))

# 3. Comparison table: Logistic Regression vs Depth-3 Tree vs Random Forest
dt3_pred = dt_3.predict(X_test)
dt3_prob = dt_3.predict_proba(X_test)[:, 1]

comparison = pd.DataFrame([
    {
        "Model": "Logistic Regression",
        "Accuracy": round(accuracy_score(y_test, y_pred), 4),
        "Recall": round(recall_score(y_test, y_pred), 4),
        "ROC-AUC": round(roc_auc_score(y_test, y_prob), 4),
    },
    {
        "Model": "Decision Tree (max_depth=3)",
        "Accuracy": round(accuracy_score(y_test, dt3_pred), 4),
        "Recall": round(recall_score(y_test, dt3_pred), 4),
        "ROC-AUC": round(roc_auc_score(y_test, dt3_prob), 4),
    },
    {
        "Model": "Random Forest (100 trees)",
        "Accuracy": round(accuracy_score(y_test, rf_pred), 4),
        "Recall": round(recall_score(y_test, rf_pred), 4),
        "ROC-AUC": round(roc_auc_score(y_test, rf_prob), 4),
    },
])
#model comparison
print("\n=== Model Comparison Table ===")
print(comparison.to_string(index=False))

=== Random Forest Metrics ===
Train accuracy: 1.0
Test accuracy:  0.7475
Recall:         0.3061
ROC-AUC:        0.7057

=== Top 3 Feature Importances ===
lead_days      0.2199
age            0.1968
distance_km    0.1947
dtype: float64

=== Model Comparison Table ===
                      Model  Accuracy  Recall  ROC-AUC
        Logistic Regression    0.7879  0.3469   0.7889
Decision Tree (max_depth=3)    0.7778  0.2245   0.6997
  Random Forest (100 trees)    0.7475  0.3061   0.7057


## Task 3.4 — Random Forest & Model Comparison Table
- Train `RandomForestClassifier(n_estimators=100, random_state=42)`, evaluate Accuracy, Recall, and ROC-AUC on the test set, and rank the top 3 predictors (`lead_days`, `age`, `distance_km`) via `feature_importances_`.
- Compile a side-by-side comparison table across Logistic Regression, Depth-3 Decision Tree, and Random Forest to select the strongest model for risk scoring.
-

Takeaway (Task 3.4): Logistic Regression looks best on the test set because it achieves the highest ROC-AUC (0.7889), highest Accuracy (0.7879), and highest Recall (0.3469) without overfitting like the default Random Forest (1.0000 training accuracy).

In [ ]:
#Task 3.5 — Tune, choose the model and the threshold, then test once
import pandas as pd
from sklearn.ensemble import RandomForestClassifier
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import f1_score, precision_score, recall_score, roc_auc_score
from sklearn.model_selection import GridSearchCV, cross_val_predict, cross_val_score
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import StandardScaler
from sklearn.tree import DecisionTreeClassifier

# (a) GridSearchCV on Random Forest (training set only)
#{"n_estimators": [100, 200], "max_depth": [3, 5, 8]}

param_grid = {"n_estimators": [100, 200], "max_depth": [3, 5, 8]}
grid = GridSearchCV(
    RandomForestClassifier(random_state=42),
    param_grid,
    #cv=5, scoring="roc_auc"
    cv=5,
    scoring="roc_auc",
)
grid.fit(X_train, y_train)
# print best_params_ and best_score_ (a cross-validated ROC-AUC)
print("=== (a) Random Forest GridSearchCV ===")
print("Best params:", grid.best_params_)
print("Best CV ROC-AUC:", round(grid.best_score_, 4))

# (b) Compare 5-fold CV ROC-AUC across all candidate models on X_train
lr_pipe = Pipeline([
    ("scaler", StandardScaler()),
    ("model", LogisticRegression(max_iter=1000)),
])
dt3_cv = DecisionTreeClassifier(max_depth=3, random_state=42)
rf_default_cv = RandomForestClassifier(n_estimators=100, random_state=42)
#cross_val_score(model, X_train, y_train, cv=5, scoring="roc_auc").mean()
cv_scores = {
    "Logistic Regression (Pipeline)": cross_val_score(
        lr_pipe, X_train, y_train, cv=5, scoring="roc_auc"
    ).mean(),
    "Decision Tree (max_depth=3)": cross_val_score(
        dt3_cv, X_train, y_train, cv=5, scoring="roc_auc"
    ).mean(),
    "Random Forest (default 100)": cross_val_score(
        rf_default_cv, X_train, y_train, cv=5, scoring="roc_auc"
    ).mean(),
    "Random Forest (tuned best)": grid.best_score_,
}

print("\n=== (b) 5-Fold Cross-Validated ROC-AUC (Training Set) ===")
for name, score in cv_scores.items():
    print(f"{name:32s} {score:.4f}")

# Pick winner dynamically based on highest CV ROC-AUC
winner_name = max(cv_scores, key=cv_scores.get)
print("\nSelected Winner:", winner_name)

# Set winner estimator
if "Logistic" in winner_name:
    winner = lr_pipe
elif "tuned" in winner_name:
    winner = grid.best_estimator_
else:
    winner = rf_default_cv

# (c) Choose threshold on out-of-fold training probabilities (no test set touched)
#cross_val_predict(model, X_train, y_train, cv=5, method="predict_proba")[:, 1]
oof_probs = cross_val_predict(
    winner, X_train, y_train, cv=5, method="predict_proba"
)[:, 1]
#compare thresholds 0.5 and 0.35 on precision
print("\n=== (c) Out-of-Fold Threshold Comparison (Training Set) ===")
best_thresh = 0.5
best_f1 = -1.0

for thresh in [0.5, 0.35]:
    oof_preds = (oof_probs >= thresh).astype(int)
    prec = precision_score(y_train, oof_preds)
    #recall, F1 and number flagged
    rec = recall_score(y_train, oof_preds)
    f1 = f1_score(y_train, oof_preds)
    flagged = int(oof_preds.sum())
    print(
        f"Threshold {thresh:.2f} -> Precision: {prec:.4f} | Recall: {rec:.4f} | "
        f"F1: {f1:.4f} | Flagged: {flagged}"
    )
    if f1 > best_f1:
        best_f1 = f1
        best_thresh = thresh

print("Chosen Threshold (higher F1):", best_thresh)

# (d) Fit winner on full training set and evaluate ONCE on the test set
winner.fit(X_train, y_train)
test_probs = winner.predict_proba(X_test)[:, 1]
test_preds = (test_probs >= best_thresh).astype(int)

print("\n=== (d) Final Single Test Set Evaluation ===")
print("Threshold used:", best_thresh)
print("Test Precision:", round(precision_score(y_test, test_preds), 4))
print("Test Recall:   ", round(recall_score(y_test, test_preds), 4))
print("Test Flagged:  ", int(test_preds.sum()), "out of", len(y_test))
print("Test ROC-AUC:  ", round(roc_auc_score(y_test, test_probs), 4))

# (e) Save test_risk_scores.csv
risk_df = pd.DataFrame({
    "appointment_id": df.loc[X_test.index, "appointment_id"].values,
    "no_show": y_test.values,
    "risk_prob": test_probs,
    "flag": test_preds,
})
risk_df.to_csv("test_risk_scores.csv", index=False)
print("\nSaved test_risk_scores.csv with shape:", risk_df.shape)
print(risk_df.head())

=== (a) Random Forest GridSearchCV ===
Best params: {'max_depth': 3, 'n_estimators': 200}
Best CV ROC-AUC: 0.7673

=== (b) 5-Fold Cross-Validated ROC-AUC (Training Set) ===
Logistic Regression (Pipeline)   0.7737
Decision Tree (max_depth=3)      0.6784
Random Forest (default 100)      0.7466
Random Forest (tuned best)       0.7673

Selected Winner: Logistic Regression (Pipeline)

=== (c) Out-of-Fold Threshold Comparison (Training Set) ===
Threshold 0.50 -> Precision: 0.5753 | Recall: 0.2485 | F1: 0.3471 | Flagged: 73
Threshold 0.35 -> Precision: 0.5130 | Recall: 0.4675 | F1: 0.4892 | Flagged: 154
Chosen Threshold (higher F1): 0.35

=== (d) Final Single Test Set Evaluation ===
Threshold used: 0.35
Test Precision: 0.5714
Test Recall:    0.5714
Test Flagged:   49 out of 198
Test ROC-AUC:   0.7889

Saved test_risk_scores.csv with shape: (198, 4)
  appointment_id  no_show  risk_prob  flag
0          A0217        0   0.021565     0
1          A0336        0   0.150917     0
2          A0508 

## Task 3.5 — Hyperparameter Tuning, Threshold Selection & Final Test Evaluation
- Tune `RandomForestClassifier` with 5-fold `GridSearchCV` (`n_estimators`: `[100, 200]`, `max_depth`: `[3, 5, 8]`), compare 5-fold CV ROC-AUC across all candidate models on `X_train`, and select the top-performing model (`Logistic Regression (Pipeline)`).
- Compare thresholds `0.50` vs. `0.35` on out-of-fold training probabilities (`cross_val_predict`) to maximize F1, evaluate the final model once on `X_test`, and export `test_risk_scores.csv`.
-

Takeaway (Task 3.5): Recall matters here because missing an actual no-show leaves an unfillable, costly doctor slot empty, whereas a false alarm only costs the front desk a one-minute phone call. In real clinic operations, management would choose the threshold based on how many follow-up calls the front desk can realistically make in an evening weighed against the cost of an empty slot.